# Diarization hyperparameter tuning

## Onderzoeksvraag

Kan de bestaande pyannote-diarization worden verbeterd door de clustering-
hyperparameters aan te passen, zonder dat bestaande sprekers onnodig worden
opgesplitst?

Er worden twee parameters onderzocht:

1. `clustering.min_cluster_size`
2. `clustering.threshold`

De huidige productie-instellingen blijven tijdens het experiment ongewijzigd.

## 1. Handmatige ground truth

Voor de evaluatie zijn korte audiofragmenten handmatig geannoteerd.

De ground truth bevat:
- starttijd
- eindtijd
- speaker-ID

De labels beschrijven alleen sprekeridentiteit en geen rol zoals DOCENT of OTHER.

In [6]:
import pandas as pd
from pathlib import Path

base = Path("../Experiments/diarization/hyperparameter_tuning")

gt4 = pd.read_csv(base / "ground_truth_testaudio4_fragment.csv")
gt4

,fragment,turn_index,start_seconds,end_seconds,speaker_id,overlap,note,transcript_hint
0,testaudio4_fragment,0,0.0,1.2,SPREKER_A,nee,NaN,"Ja, als ik jou nog één keer hoor."
1,testaudio4_fragment,1,1.2,2.4,SPREKER_B,nee,NaN,Dat vind ik echt niet eerlijk.
2,testaudio4_fragment,2,2.4,5.1,SPREKER_A,nee,NaN,"Nou, ik hoor jouw stem nou, toch?"
3,testaudio4_fragment,3,5.1,5.8,SPREKER_B,nee,NaN,"Ja, dat klopt."
4,testaudio4_fragment,4,5.8,6.4,SPREKER_A,nee,NaN,"Ja, nou."
...,...,...,...,...,...,...,...,...
82,testaudio4_fragment,82,116.0,116.6,SPREKER_B,nee,rumoer,Zeker.
83,testaudio4_fragment,83,116.6,117.7,SPREKER_A,nee,rumoer,"Nee, dat heb ik echt wel."
84,testaudio4_fragment,84,117.7,118.2,SPREKER_A,nee,rumoer,Nee.
85,testaudio4_fragment,85,118.2,119.2,SPREKER_A,nee,rumoer,"Oké, luister."


In [9]:
def ground_truth_summary(df):
    df = df.copy()
    df["duration"] = df["end_seconds"] - df["start_seconds"]

    return (
        df.groupby("speaker_id")
          .agg(
              turns=("speaker_id", "size"),
              seconds=("duration", "sum")
          )
          .round(2)
    )

ground_truth_summary(gt4)

,turns,seconds
speaker_id,,
SPREKER_A,50,72.0
SPREKER_B,31,36.9
SPREKER_C,5,7.3
SPREKER_D,1,0.8


In [7]:
gt2 = pd.read_csv(base / "ground_truth_testaudio2_fragment.csv")
gt2

,fragment,turn_index,start_seconds,end_seconds,speaker_id,overlap,note,transcript_hint
0,testaudio2_fragment,0,0.0,2.0,SPREKER_A,nee,NaN,"Vier, Rick. Zegt u het maar."
1,testaudio2_fragment,1,2.0,4.0,SPREKER_A,nee,NaN,"Ja, kijk maar even in je schrift op de vier."
2,testaudio2_fragment,2,4.0,5.0,SPREKER_B,nee,NaN,Ruud heeft een brommer.
3,testaudio2_fragment,3,5.0,7.0,SPREKER_A,nee,NaN,"De tekst gaat toch niet over brommer, of wel?"
4,testaudio2_fragment,4,7.0,9.0,SPREKER_B,nee,NaN,"Nee, maar het gaat ook niet om de brommer."
5,testaudio2_fragment,5,9.0,11.0,SPREKER_A,ja,spreker b praat er nog in de achtergrond doorheen,"Maar Ruud heeft een probleem, dat is de vraag."
6,testaudio2_fragment,6,11.0,13.0,SPREKER_A,nee,NaN,Wat voor probleem heeft-ie?
7,testaudio2_fragment,7,13.0,14.0,SPREKER_B,nee,spreker slaat op de tafel,Met zijn vrouw.
8,testaudio2_fragment,8,14.0,15.0,SPREKER_A,nee,NaN,"Nee, klopt niet."
9,testaudio2_fragment,9,15.0,16.0,SPREKER_B,nee,NaN,"Ik bedoel, met zijn hond."


In [10]:
def ground_truth_summary(df):
    df = df.copy()
    df["duration"] = df["end_seconds"] - df["start_seconds"]

    return (
        df.groupby("speaker_id")
          .agg(
              turns=("speaker_id", "size"),
              seconds=("duration", "sum")
          )
          .round(2)
    )

ground_truth_summary(gt2)

,turns,seconds
speaker_id,,
SPREKER_A,13,23.0
SPREKER_B,6,7.0


## 2. Experiment: `min_cluster_size`

`min_cluster_size` bepaalt hoe groot een cluster van stemfragmenten moet zijn
voordat pyannote deze als zelfstandige spreker behoudt.

Hypothese:
een lagere waarde kan korte sprekers beter terugvinden, maar kan tegelijkertijd
bestaande sprekers opsplitsen in meerdere clusters.

Geteste waarden:

- 12 — huidige instelling
- 10
- 8
- 6
- 3

In [11]:
import json

with open(
    base / "results_min_cluster_size_testaudio4_fragment_run1_12-6-3.json",
    encoding="utf-8"
) as f:
    run1 = json.load(f)

with open(
    base / "results_min_cluster_size_testaudio4_fragment.json",
    encoding="utf-8"
) as f:
    run2 = json.load(f)

results4 = {**run1, **run2}
results4

{'12': {'gt_speakers': 4,
  'pyannote_speakers_found': 2,
  'overall_correct_pct': 81.0,
  'per_gt_speaker': {'SPREKER_A': {'seconds': 72.0,
    'matched_pyannote_speaker': 'SPEAKER_00',
    'covered_pct': 86.9},
   'SPREKER_B': {'seconds': 36.9,
    'matched_pyannote_speaker': 'SPEAKER_01',
    'covered_pct': 87.3},
   'SPREKER_C': {'seconds': 7.3,
    'matched_pyannote_speaker': None,
    'covered_pct': 0.0},
   'SPREKER_D': {'seconds': 0.8,
    'matched_pyannote_speaker': None,
    'covered_pct': 0.0}},
  'pyannote_turns': 63,
  'runtime_seconds': 87.92},
 '6': {'gt_speakers': 4,
  'pyannote_speakers_found': 6,
  'overall_correct_pct': 75.7,
  'per_gt_speaker': {'SPREKER_A': {'seconds': 72.0,
    'matched_pyannote_speaker': 'SPEAKER_01',
    'covered_pct': 82.4},
   'SPREKER_B': {'seconds': 36.9,
    'matched_pyannote_speaker': 'SPEAKER_05',
    'covered_pct': 67.2},
   'SPREKER_C': {'seconds': 7.3,
    'matched_pyannote_speaker': 'SPEAKER_02',
    'covered_pct': 50.7},
   'SPREKER_

In [12]:
rows = []

for value, result in results4.items():
    row = {
        "min_cluster_size": int(value),
        "gevonden_sprekers": result["pyannote_speakers_found"],
        "totaal_juist_pct": result["overall_correct_pct"],
    }

    for speaker, stats in result["per_gt_speaker"].items():
        row[f"{speaker}_pct"] = stats["covered_pct"]

    rows.append(row)

df_results4 = (
    pd.DataFrame(rows)
      .sort_values("min_cluster_size", ascending=False)
)

df_results4

,min_cluster_size,gevonden_sprekers,totaal_juist_pct,SPREKER_A_pct,SPREKER_B_pct,SPREKER_C_pct,SPREKER_D_pct
0,12,2,81.0,86.9,87.3,0.0,0.0
3,10,2,81.0,86.9,87.3,0.0,0.0
4,8,4,83.2,82.4,84.0,97.3,0.0
1,6,6,75.7,82.4,67.2,50.7,100.0
2,3,6,75.7,82.4,67.2,50.7,100.0


### Resultaat testaudio4

De huidige instelling `12` vindt slechts 2 van de 4 echte sprekers.

Het verlagen naar 6 of 3 zorgt ervoor dat korte sprekers beter worden gevonden,
maar leidt ook tot over-segmentatie: pyannote vindt 6 sprekers terwijl er 4 echte
sprekers aanwezig zijn.

`min_cluster_size = 8` geeft op dit fragment de beste balans:

- 4 echte sprekers → 4 gevonden sprekers
- totale overeenkomst: 83,2%
- spreker C wordt voor 97,3% teruggevonden

De winst ten opzichte van de huidige instelling is echter beperkt:
81,0% → 83,2%.

Daarom moet deze instelling op een tweede fragment worden gecontroleerd.

## 3. Validatie van `min_cluster_size = 8`

De waarde 8 is gekozen op basis van testaudio4.

Om te voorkomen dat de instelling alleen op dat fragment goed werkt, wordt
deze zonder verdere tuning getest op `testaudio2_fragment`.

Hier worden alleen de huidige waarde 12 en kandidaat 8 vergeleken.

In [13]:
with open(
    base / "results_min_cluster_size_testaudio2_fragment.json",
    encoding="utf-8"
) as f:
    results2 = json.load(f)

rows = []

for value, result in results2.items():
    row = {
        "min_cluster_size": int(value),
        "gevonden_sprekers": result["pyannote_speakers_found"],
        "totaal_juist_pct": result["overall_correct_pct"],
    }

    for speaker, stats in result["per_gt_speaker"].items():
        row[f"{speaker}_pct"] = stats["covered_pct"]

    rows.append(row)

df_results2 = pd.DataFrame(rows)
df_results2

,min_cluster_size,gevonden_sprekers,totaal_juist_pct,SPREKER_A_pct,SPREKER_B_pct
0,12,2,78.0,96.5,17.1
1,8,2,78.0,96.5,17.1


### Resultaat testaudio2

Op testaudio2 maakt `min_cluster_size = 8` geen verschil.

Beide instellingen:

- vinden 2 sprekers;
- behalen 78,0% totale overeenkomst;
- herkennen spreker A voor 96,5%;
- herkennen spreker B slechts voor 17,1%.

Het probleem in dit fragment lijkt daarom niet veroorzaakt te worden door
`min_cluster_size`.

De pipeline vindt wel twee speakerclusters, maar een groot deel van de spraak
van spreker B wordt aan spreker A toegewezen.

Daarom wordt `min_cluster_size` niet aangepast in de productiepipeline.

## Aanvullende test: clustering.threshold

Omdat `min_cluster_size` het samenvoegingsprobleem in testaudio2 niet oploste,
is kort getest of `clustering.threshold` dit beter kon beïnvloeden.

Geteste waarden:
- 0.70 — huidig
- 0.60
- 0.50

In [14]:
with open(base / "results_threshold_testaudio2_fragment.json", encoding="utf-8") as f:
    threshold_audio2 = json.load(f)

with open(base / "results_threshold_testaudio4_fragment.json", encoding="utf-8") as f:
    threshold_audio4 = json.load(f)

In [15]:
rows = []

for threshold in ["0.704565", "0.6", "0.5"]:
    rows.append({
        "threshold": threshold,
        "testaudio4_sprekers": threshold_audio4[threshold]["pyannote_speakers_found"],
        "testaudio4_totaal": threshold_audio4[threshold]["overall_correct_pct"],
        "testaudio2_sprekers": threshold_audio2[threshold]["pyannote_speakers_found"],
        "testaudio2_totaal": threshold_audio2[threshold]["overall_correct_pct"],
    })

df_threshold = pd.DataFrame(rows)

df_threshold

,threshold,testaudio4_sprekers,testaudio4_totaal,testaudio2_sprekers,testaudio2_totaal
0,0.704565,2,81.0,2,78.0
1,0.6,2,81.0,3,57.7
2,0.5,2,66.5,3,57.7


In [17]:
df_threshold = df_threshold.rename(columns={
    "threshold": "Threshold",
    "testaudio4_sprekers": "Audio 4 - sprekers",
    "testaudio4_totaal": "Audio 4 - totaal (%)",
    "testaudio2_sprekers": "Audio 2 - sprekers",
    "testaudio2_totaal": "Audio 2 - totaal (%)",
})

df_threshold

,Threshold,Audio 4 - sprekers,Audio 4 - totaal (%),Audio 2 - sprekers,Audio 2 - totaal (%)
0,0.704565,2,81.0,2,78.0
1,0.6,2,81.0,3,57.7
2,0.5,2,66.5,3,57.7


## 5. Eindconclusie

### `min_cluster_size`

Een lagere `min_cluster_size` kan korte sprekers beter zichtbaar maken.

Op testaudio4 gaf waarde 8 een kleine verbetering en werd de eerder gemiste
spreker C bijna volledig teruggevonden.

Deze verbetering generaliseerde echter niet naar testaudio2.

Daarom is er onvoldoende bewijs om de productie-instelling van
`min_cluster_size` aan te passen.


### Besluit voor de pipeline

Alleen instellingen die op meerdere fragmenten aantoonbaar verbeteren zonder
nieuwe diarization-problemen te veroorzaken, worden overwogen voor de
productiepipeline. 
Helaas geven deze tests weer dat aanpassingen in de instellingen niet (veel) verschil maken voor de nauwkeurigheid. De conclusie is dus dat er geen verandering zal plaatsvinden. 